# Kaggle submission: XGBoost trained on all labeled cases

What is produced: two Kaggle selections, each predictions for the 725
hidden test cases (Run No 2923-3647, starts 2023-01-01 to 2024-12-25) in
the exact `sample_submission.csv` format (filled by column name from that
file as the template):

1. `../output.csv` — the baseline 193-feature pipeline (scored 0.98722).
2. `../output2.csv` — the same pipeline plus the volpath (reservoir volume
   trajectory, day 1-7) and cyclic (hour/day-of-week sin/cos) feature
   groups. Measured val-2022 micro AUC 0.9806 vs baseline 0.9802 — within
   noise, so this is a diversification pick for the two-prediction format
   (the better of the two selected counts), not an improvement claim.

Protocol: the baseline pipeline unchanged — 14 per-generator XGBoost
models on the 193 features, trained on ALL 2922 labeled cases
(2015-2022), early stopping on the last 90 days of the train period
(case starts 2022-10-03 to 2022-12-31, in-sample per the measured
train_tail protocol), seed 0, lr 0.05, 400 trees.

Data coverage note: the provided files exactly cover the test window —
the last test case (start 2024-12-25) needs the terminal water value at
start + 168h = 2025-01-01 00:00, which is the last row of the water-value
and volume files. No NaN gaps are expected; the notebook prints NaN
counts as a check.

Printed sanity checks: case counts, per-generator tree counts, NaN
counts in the test features and in the output, and the mean predicted
ON-rate per generator for the test years next to its train ON-rate
(the EDA measured strong year drift, so this is the drift diagnostic).

In [1]:
import gc
import numpy as np
import pandas as pd
import xgboost as xgb
import yaml

HOURS = 168

In [2]:
DATA = "../data/kernel"
EXT = "../data/extended"

price = pd.read_csv(f"{DATA}/Historical_day_ahead_price_2015_2025.csv")
inflow = pd.read_csv(f"{DATA}/Historical_inflow_1958_2025.csv")
volume = pd.read_csv(f"{DATA}/Historical_volume_2015_2024.csv")
water = pd.read_csv(f"{DATA}/Synthetic_water_value_2015_2024.csv")
target = pd.read_csv(f"{DATA}/Unit_commitment_decisions.csv")
min_flow = pd.read_csv(f"{EXT}/Constraint_min_flow.csv")
min_volume = pd.read_csv(f"{EXT}/Constraint_min_volume.csv")
topology = yaml.safe_load(open(f"{EXT}/Tokke_Vinje_topology.yaml"))


def indexed(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], utc=True, format="mixed")
    return df.sort_values("date").set_index("date")


price_h = indexed(price)["Day-ahead price (EUR/MWh)"].resample("1h").mean()
inf_wide = indexed(inflow)
vol_wide = indexed(volume)
wv_wide = indexed(water)
mf_wide = indexed(min_flow)
mv_wide = indexed(min_volume)

RESERVOIRS = list(vol_wide.columns)
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}

target["starttime"] = pd.to_datetime(target["starttime"], utc=True, format="mixed")
target = target.sort_values("starttime").reset_index(drop=True)

GENERATORS = sorted(
    {c.removeprefix("result_committed_").rsplit("_t", 1)[0]
     for c in target.columns if c.startswith("result_committed")}
)

connections = topology["connections"]


def supplying_reservoirs(name, kind):
    if kind == "reservoir":
        return {name}
    out = set()
    for e in connections:
        if e["to"] == name and e["to_type"] == kind and e["from_type"] in {"reservoir", "tunnel"}:
            out |= supplying_reservoirs(e["from"], e["from_type"])
    return out


FEEDERS = {}
for gen in GENERATORS:
    plant = next(e["to"] for e in connections
                 if e["from"] == gen and e["from_type"] == "generator" and e["to_type"] == "plant")
    FEEDERS[gen] = sorted(supplying_reservoirs(plant, "plant"))

print("labeled cases:", len(target), " generators:", len(GENERATORS))

labeled cases: 2922  generators: 14


## Feature engineering

What is produced: the unchanged 193-column feature set, copied verbatim
from `notebooks/xgboost_baseline.ipynb`, applied to the train cases
(all 2922), the stop window (last 90 days of 2022), and the test cases
from `prediction_mapping.csv`.

In [3]:
def lookup(frame, timestamps, columns=None):
    f = frame if columns is None else frame[columns]
    return f.reindex(pd.DatetimeIndex(timestamps)).to_numpy(dtype=np.float32)


def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    n = len(starts)
    ts = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h"
    )
    price = lookup(price_h, ts).reshape(n, HOURS)
    initial = lookup(vol_wide, starts, RESERVOIRS)
    terminal = lookup(wv_wide, starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflow = lookup(inf_wide, ts).reshape(n, HOURS, -1)
    minflow = lookup(mf_wide, ts).reshape(n, HOURS, -1)
    minvol = lookup(mv_wide, ts).reshape(n, HOURS, -1)

    F = {}

    def add(name, v):
        v = np.asarray(v, dtype=np.float32)
        if v.shape == (n,):
            v = np.repeat(v, HOURS)
        elif v.shape == (n, HOURS):
            v = v.reshape(-1)
        F[name] = v

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", ts.hour.to_numpy())
    add("day_of_week", ts.dayofweek.to_numpy())
    add("month", ts.month.to_numpy())
    add("doy_sin", np.sin(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))
    add("doy_cos", np.cos(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))

    add("price", price)
    for stat, v in (("mean", price.mean(1)), ("std", price.std(1)),
                    ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", v)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, v in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(v, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    for off in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{off:+d}", price[:, np.clip(np.arange(HOURS) + off, 0, HOURS - 1)])

    for j, res in enumerate(RESERVOIRS):
        add(f"init_vol__{res}", initial[:, j])
        add(f"init_frac__{res}", initial[:, j] / CAPACITY[res])
        add(f"term_val__{res}", terminal[:, j])
        add(f"price_minus_val__{res}", price - terminal[:, j, None])

    for j, name in enumerate(inf_wide.columns):
        flow = inflow[:, :, j]
        cum = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cum_Mm3__{name}", cum)
        add(f"inflow_rem_Mm3__{name}", cum[:, -1:] - cum)

    for j, name in enumerate(mf_wide.columns):
        add(f"minflow__{name}", minflow[:, :, j])
        add(f"minflow_week_max__{name}", minflow[:, :, j].max(1))
    for j, name in enumerate(mv_wide.columns):
        add(f"minvol__{name}", minvol[:, :, j])
        add(f"minvol_week_max__{name}", minvol[:, :, j].max(1))
        add(f"init_margin_to_minvol__{name}", initial[:, RESERVOIRS.index(name), None] - minvol[:, :, j])

    return pd.DataFrame(F, dtype=np.float32)


def generator_features(base, gen):
    X = base.copy()
    feeders = FEEDERS[gen]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_init = sum(base[f"init_vol__{r}"] for r in feeders)
    value = sum(base[f"term_val__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cum = sum(base[f"inflow_cum_Mm3__{r}"] for r in feeders)
    rem = sum(base[f"inflow_rem_Mm3__{r}"] for r in feeders)

    X["local_init_frac"] = total_init / capacity
    X["local_term_val"] = value
    X["local_price_minus_val"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cum_frac"] = cum / capacity
    X["local_rem_frac"] = rem / capacity
    X["local_available_frac"] = (total_init + cum + rem) / capacity
    return X


def labels_for(cases, gen):
    cols = [f"result_committed_{gen}_t{t}" for t in range(HOURS)]
    return cases[cols].to_numpy(dtype=np.uint8).reshape(-1)


train_cases = target.copy()
stop_cases = train_cases[
    train_cases["starttime"] >= train_cases["starttime"].max() - pd.Timedelta(days=90)
].reset_index(drop=True)

mapping = pd.read_csv("../prediction_mapping.csv")
mapping["starttime"] = pd.to_datetime(mapping["starttime"], utc=True, format="mixed")
test_cases = mapping.sort_values("starttime").reset_index(drop=True)

print("train cases:", len(train_cases), " stop cases:", len(stop_cases),
      " test cases:", len(test_cases))

Xtr_base = feature_matrix(train_cases)
Xst_base = feature_matrix(stop_cases)
Xte_base = feature_matrix(test_cases)
print("train matrix:", Xtr_base.shape, " test matrix:", Xte_base.shape)
print("NaNs in test features:", int(Xte_base.isna().to_numpy().sum()))

train cases: 2922  stop cases: 91  test cases: 725
train matrix: (490896, 186)  test matrix: (121800, 186)
NaNs in test features: 0


## Fit on all labeled cases, predict the test cases

What is produced: one XGBoost per generator trained on all 2922 labeled
cases with the train_tail stop window (last 90 days of 2022, in-sample),
predicting the 725 test cases. Printed: per-generator tree count and the
mean predicted test ON-rate next to the train ON-rate.

In [4]:
test_preds = {}
rows = []
for gen in GENERATORS:
    Xtr = generator_features(Xtr_base, gen)
    Xst = generator_features(Xst_base, gen)
    Xte = generator_features(Xte_base, gen)
    ytr = labels_for(train_cases, gen)
    yst = labels_for(stop_cases, gen)

    if np.unique(ytr).size == 1:
        test_preds[gen] = np.full(len(test_cases) * HOURS, float(ytr[0]), dtype=np.float32)
        rows.append({"generator": gen, "trees": 0,
                     "train_on": ytr.mean(), "test_pred": test_preds[gen].mean()})
    else:
        clf = xgb.XGBClassifier(
            n_estimators=400, learning_rate=0.05, max_depth=7,
            subsample=0.8, colsample_bytree=0.8, tree_method="hist",
            eval_metric="auc", early_stopping_rounds=25, random_state=0,
        )
        clf.fit(Xtr, ytr, eval_set=[(Xst, yst)], verbose=False)
        test_preds[gen] = clf.predict_proba(Xte)[:, 1].astype(np.float32)
        rows.append({"generator": gen, "trees": clf.best_iteration + 1,
                     "train_on": ytr.mean(), "test_pred": test_preds[gen].mean()})
        del clf
    del Xtr, Xst, Xte
    gc.collect()

pd.DataFrame(rows).set_index("generator").round(3)

,trees,train_on,test_pred
generator,,,
Byrte_G1,400,0.542,0.591
Haukeli_G1,400,0.547,0.359
Hogga_G1,399,0.595,0.608
Kjela_G1,400,0.471,0.429
Lio_G1,400,0.747,0.755
Songa_G1,400,0.485,0.534
Tokke_G1,400,0.671,0.588
Tokke_G2,399,0.478,0.414
Tokke_G3,400,0.564,0.497


## Write `../output.csv`

What is produced: `../output.csv` in the exact `sample_submission.csv`
format — the template file supplies the row order (Run No) and column
names; each `result_committed_<gen>_t<h>` column is filled from the
predictions via the Run No -> starttime mapping. Printed: output shape,
NaN count, and head of the file.

In [5]:
template = pd.read_csv("../sample_submission.csv")
start_by_run = test_cases.set_index("Run No")["starttime"]
pos_by_start = {ts: i for i, ts in enumerate(test_cases["starttime"])}

out = template.copy()
rows_idx = np.array([pos_by_start[start_by_run[run]] for run in out["Run No"]])
for gen in GENERATORS:
    p = test_preds[gen].reshape(len(test_cases), HOURS)
    for t in range(HOURS):
        col = f"result_committed_{gen}_t{t}"
        out[col] = p[rows_idx, t]

print("output shape:", out.shape)
print("NaNs in output:", int(out.isna().to_numpy().sum()))
print("value range:", float(out.iloc[:, 1:].to_numpy().min()),
      "-", float(out.iloc[:, 1:].to_numpy().max()))
out.to_csv("../output.csv", index=False)
print("saved ../output.csv")
out.head()

output shape: (725, 2353)
NaNs in output: 0
value range: 3.1249209087036434e-07 - 0.9999997615814209
saved ../output.csv


,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4,result_committed_Hogga_G1_t5,result_committed_Hogga_G1_t6,result_committed_Hogga_G1_t7,result_committed_Hogga_G1_t8,...,result_committed_Vesle Kjela_G1_t158,result_committed_Vesle Kjela_G1_t159,result_committed_Vesle Kjela_G1_t160,result_committed_Vesle Kjela_G1_t161,result_committed_Vesle Kjela_G1_t162,result_committed_Vesle Kjela_G1_t163,result_committed_Vesle Kjela_G1_t164,result_committed_Vesle Kjela_G1_t165,result_committed_Vesle Kjela_G1_t166,result_committed_Vesle Kjela_G1_t167
0,2923,0.336279,0.187658,0.074364,0.026659,0.011690,0.010514,0.005144,0.014445,0.096588,...,0.999883,0.999787,0.999766,0.999718,0.999819,0.999857,0.999902,0.999750,0.999660,0.930918
1,2924,0.923033,0.912714,0.912714,0.938953,0.898937,0.944032,0.983749,0.990373,0.993099,...,0.999791,0.999642,0.999562,0.999586,0.999638,0.999627,0.999641,0.999667,0.999769,0.999763
2,2925,0.997649,0.997562,0.997677,0.998461,0.999221,0.999356,0.999805,0.999808,0.999813,...,0.999905,0.999892,0.999863,0.999853,0.999853,0.999807,0.999818,0.999876,0.999888,0.999927
3,2926,0.999931,0.999909,0.999918,0.999940,0.999957,0.999966,0.999975,0.999979,0.999990,...,0.999918,0.999875,0.999869,0.999845,0.999871,0.999889,0.999841,0.999879,0.999796,0.999736
4,2927,0.999896,0.999870,0.999850,0.999936,0.999937,0.999947,0.999951,0.999955,0.999969,...,0.999918,0.999888,0.999858,0.999834,0.999850,0.999885,0.999872,0.999906,0.999914,0.999916


## Second prediction: volpath + cyclic variant

What is produced: `../output2.csv` — the same full-train pipeline with
the volpath group (per-reservoir volume fraction at each of the 7 day
boundaries of the case week, plus per-generator local capacity-weighted
fractions) and the cyclic group (hour-of-day and day-of-week sin/cos)
added to the 193 baseline features. Under the clean val-2022 protocol
this variant measured 0.9806 vs the baseline's 0.9802 (within noise;
see `notebooks/xgboost_volpath.ipynb`). The submission format allows
two selected predictions and the better counts, so this is the second
selection. Printed: extra matrix shapes, NaN counts, per-generator
tree counts and test ON-rates.

In [6]:
def extra_features(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    n = len(starts)
    ts = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h"
    )
    E = {}

    def add(name, v):
        v = np.asarray(v, dtype=np.float32)
        if v.shape == (n,):
            v = np.repeat(v, HOURS)
        elif v.shape == (n, HOURS):
            v = v.reshape(-1)
        E[name] = v

    def add_day(name, v, d):
        week = np.zeros((n, 7), dtype=np.float32)
        week[:, d - 1] = v
        E[name] = np.repeat(week, 24, axis=1).reshape(-1)

    for d in range(1, 8):
        vols = lookup(vol_wide, starts + pd.Timedelta(days=d), RESERVOIRS)
        for j, res in enumerate(RESERVOIRS):
            add_day(f"volfrac_d{d}__{res}", vols[:, j] / CAPACITY[res], d)

    hod = ts.hour.to_numpy()
    dow = ts.dayofweek.to_numpy()
    add("hod_sin", np.sin(2 * np.pi * hod / 24))
    add("hod_cos", np.cos(2 * np.pi * hod / 24))
    add("dow_sin", np.sin(2 * np.pi * dow / 7))
    add("dow_cos", np.cos(2 * np.pi * dow / 7))

    return pd.DataFrame(E, dtype=np.float32)


EXTRA_TRAIN = extra_features(train_cases)
EXTRA_STOP = extra_features(stop_cases)
EXTRA_TEST = extra_features(test_cases)
print("extra matrices:", EXTRA_TRAIN.shape, EXTRA_STOP.shape, EXTRA_TEST.shape)
print("NaNs in extras:", int(EXTRA_TRAIN.isna().to_numpy().sum()),
      int(EXTRA_STOP.isna().to_numpy().sum()),
      int(EXTRA_TEST.isna().to_numpy().sum()))

extra matrices: (490896, 123) (15288, 123) (121800, 123)
NaNs in extras: 0 0 0


In [7]:
def variant_cols(E, gen):
    feeders = FEEDERS[gen]
    cap = sum(CAPACITY[r] for r in feeders)
    X = E[[f"volfrac_d{d}__{r}" for d in range(1, 8) for r in RESERVOIRS]].copy()
    for d in range(1, 8):
        X[f"local_volfrac_d{d}"] = sum(
            E[f"volfrac_d{d}__{r}"] * CAPACITY[r] for r in feeders) / cap
    return pd.concat([X, E[["hod_sin", "hod_cos", "dow_sin", "dow_cos"]]], axis=1)


test_preds2 = {}
rows = []
for gen in GENERATORS:
    Xtr = pd.concat([generator_features(Xtr_base, gen), variant_cols(EXTRA_TRAIN, gen)], axis=1)
    Xst = pd.concat([generator_features(Xst_base, gen), variant_cols(EXTRA_STOP, gen)], axis=1)
    Xte = pd.concat([generator_features(Xte_base, gen), variant_cols(EXTRA_TEST, gen)], axis=1)
    ytr = labels_for(train_cases, gen)
    yst = labels_for(stop_cases, gen)

    if np.unique(ytr).size == 1:
        test_preds2[gen] = np.full(len(test_cases) * HOURS, float(ytr[0]), dtype=np.float32)
        rows.append({"generator": gen, "trees": 0,
                     "train_on": ytr.mean(), "test_pred": test_preds2[gen].mean()})
    else:
        clf = xgb.XGBClassifier(
            n_estimators=400, learning_rate=0.05, max_depth=7,
            subsample=0.8, colsample_bytree=0.8, tree_method="hist",
            eval_metric="auc", early_stopping_rounds=25, random_state=0,
        )
        clf.fit(Xtr, ytr, eval_set=[(Xst, yst)], verbose=False)
        test_preds2[gen] = clf.predict_proba(Xte)[:, 1].astype(np.float32)
        rows.append({"generator": gen, "trees": clf.best_iteration + 1,
                     "train_on": ytr.mean(), "test_pred": test_preds2[gen].mean()})
        del clf
    del Xtr, Xst, Xte
    gc.collect()

pd.DataFrame(rows).set_index("generator").round(3)

,trees,train_on,test_pred
generator,,,
Byrte_G1,400,0.542,0.591
Haukeli_G1,400,0.547,0.353
Hogga_G1,400,0.595,0.608
Kjela_G1,399,0.471,0.429
Lio_G1,400,0.747,0.754
Songa_G1,400,0.485,0.536
Tokke_G1,400,0.671,0.590
Tokke_G2,400,0.478,0.416
Tokke_G3,400,0.564,0.497


In [8]:
out2 = template.copy()
for gen in GENERATORS:
    p = test_preds2[gen].reshape(len(test_cases), HOURS)
    for t in range(HOURS):
        col = f"result_committed_{gen}_t{t}"
        out2[col] = p[rows_idx, t]

print("output2 shape:", out2.shape)
print("NaNs in output2:", int(out2.isna().to_numpy().sum()))
print("value range:", float(out2.iloc[:, 1:].to_numpy().min()),
      "-", float(out2.iloc[:, 1:].to_numpy().max()))
out2.to_csv("../output2.csv", index=False)
print("saved ../output2.csv")
out2.head()

output2 shape: (725, 2353)
NaNs in output2: 0
value range: 4.157009527716582e-07 - 0.9999997615814209
saved ../output2.csv


,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4,result_committed_Hogga_G1_t5,result_committed_Hogga_G1_t6,result_committed_Hogga_G1_t7,result_committed_Hogga_G1_t8,...,result_committed_Vesle Kjela_G1_t158,result_committed_Vesle Kjela_G1_t159,result_committed_Vesle Kjela_G1_t160,result_committed_Vesle Kjela_G1_t161,result_committed_Vesle Kjela_G1_t162,result_committed_Vesle Kjela_G1_t163,result_committed_Vesle Kjela_G1_t164,result_committed_Vesle Kjela_G1_t165,result_committed_Vesle Kjela_G1_t166,result_committed_Vesle Kjela_G1_t167
0,2923,0.242907,0.178437,0.077636,0.019730,0.009285,0.008467,0.004098,0.011439,0.082225,...,0.999839,0.999825,0.999752,0.999645,0.999750,0.999785,0.999783,0.999667,0.999625,0.867981
1,2924,0.928086,0.929354,0.929354,0.941543,0.935373,0.936050,0.976737,0.991939,0.995219,...,0.999772,0.999732,0.999730,0.999701,0.999739,0.999753,0.999744,0.999785,0.999850,0.999775
2,2925,0.997566,0.997444,0.997339,0.998029,0.999211,0.999343,0.999516,0.999565,0.999585,...,0.999892,0.999890,0.999839,0.999827,0.999835,0.999808,0.999816,0.999872,0.999887,0.999898
3,2926,0.999864,0.999834,0.999856,0.999885,0.999943,0.999970,0.999982,0.999984,0.999989,...,0.999920,0.999913,0.999861,0.999845,0.999862,0.999869,0.999873,0.999914,0.999789,0.999696
4,2927,0.999740,0.999722,0.999677,0.999891,0.999916,0.999930,0.999943,0.999946,0.999959,...,0.999873,0.999857,0.999825,0.999839,0.999804,0.999809,0.999847,0.999890,0.999905,0.999911
